# Quota di sezioni critiche per provincia

**Obiettivo di questo notebook:** individuare le province italiane piu'
critiche osservando la **quota di sezioni critiche sul totale** (sezioni con
`gap_score` sopra la soglia del gomito, diviso il totale delle sezioni
eleggibili della provincia).

**Perche' non la mediana su tutte le sezioni:** calcolando la mediana del
`gap_score` su tutte le sezioni eleggibili di ciascuna provincia, nessuna
provincia supera la soglia del gomito (0.429). Questo non e' un caso: la
soglia e' stata calibrata sulla distribuzione delle singole sezioni (~350.000
punti), mentre una mediana calcolata su centinaia o migliaia di sezioni per
provincia comprime naturalmente la variabilita' — per superare 0.429 in
mediana servirebbe che la *maggioranza* delle sezioni di quella provincia
fosse individualmente estrema, una condizione molto piu' rara e diversa da
"avere alcune sezioni critiche". La quota di sezioni critiche, al contrario,
cattura direttamente la concentrazione del fenomeno senza questo effetto di
compressione: la mediana resta comunque in tabella come indicatore di
contesto.

Si appoggia su due notebook precedenti del progetto:

1. `gap_score_DEFINITIVO.ipynb` (cartella Drive `4_GAP_SCORE`) — calcola il
   `gap_score` per ogni sezione di censimento eleggibile.
2. `Ranking_Sezioni_Critiche_GAP_Score.ipynb` — individua, con il metodo del
   gomito applicato alla coda dei `gap_score` positivi, la soglia
   **gap_score > 0.429**.

Struttura:
1. Richiamo alla metodologia e caricamento dati
2. Soglia del gomito e sezioni critiche
3. Quota di sezioni critiche per provincia (e mediana di contesto)
4. Grafico riassuntivo: ranking delle province per quota di sezioni critiche
5. Distribuzione del gap_score, per provincia
6. Province con pochi dati o senza sezioni critiche
7. Esportazione


## 1. Richiamo alla metodologia e caricamento dati

Il `gap_score` (definito in `gap_score_DEFINITIVO.ipynb`) e' una differenza fra
due ranghi percentile nazionali:

$$\boxed{\;\text{GAP}_i = \text{rank\%}(D_i)\; -\; \text{rank\%}(S_i)\;}$$

dove $D_i$ e' la domanda effettiva di ricarica pubblica e $S_i$ l'accessibilita'
dell'offerta, calcolati solo sulle sezioni eleggibili. E' quindi sempre in
$[-1, +1]$: vicino a **+1** = deserto (tanta domanda, offerta minima), vicino a
**0** = equilibrata, vicino a **-1** = ben servita. Le sezioni non eleggibili
hanno `gap_score = NaN`.

Qui carichiamo il file completo (non il solo CSV `SEZ2011 + gap_score`) perche'
ci serve la colonna `PROVINCIA` per l'aggregazione geografica.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.dpi"] = 110

# Percorso assoluto: in un notebook VS Code il kernel parte con la working
# directory dell'eseguibile di VS Code, quindi ne' __file__ ne' Path.cwd()
# sono affidabili qui. Se sposti il parquet altrove, aggiorna questo percorso.
CARTELLA_PROGETTO = Path(r"C:\Users\fasanelli michele\OneDrive\Desktop\Contesto lavoro di gruppo ETL")
PATH = CARTELLA_PROGETTO / "sezioni_gap_score_DEFINITIVO.parquet"

# Usiamo il parquet (non il CSV) perche' serve la colonna PROVINCIA, assente
# nell'export CSV finale della pipeline (gap_score_definitivo.csv ha solo
# SEZ2011 + gap_score).
df = pd.read_parquet(PATH, columns=["SEZ2011", "PROVINCIA", "gap_score"])
print(f"Righe totali nel file:         {len(df):,}")
df = df.dropna(subset=["gap_score"]).copy()
print(f"Sezioni eleggibili (non NaN):  {len(df):,}")
print(f"Province presenti:             {df['PROVINCIA'].nunique()}")
df.head()


## 2. Soglia del gomito e sezioni critiche

`Ranking_Sezioni_Critiche_GAP_Score.ipynb` ha applicato il metodo del gomito
alla coda dei `gap_score` positivi (i potenziali deserti), trovando:

$$\text{SOGLIA\_GOMITO} = 0{,}429$$

Qui usiamo la soglia per marcare quali sezioni sono "critiche": la quota di
queste per provincia e' la metrica su cui costruiamo il ranking (sezione 3).


In [ ]:
# Soglia dal notebook Ranking_Sezioni_Critiche_GAP_Score.ipynb.
SOGLIA_GOMITO = 0.429

df["critica"] = df["gap_score"] > SOGLIA_GOMITO
critiche = df[df["critica"]]
quota_nazionale = df["critica"].mean()
print(f"Sezioni sopra la soglia del gomito (gap_score > {SOGLIA_GOMITO}): {len(critiche):,} "
      f"({quota_nazionale:.1%} del totale eleggibile)")


## 3. Quota di sezioni critiche per provincia (e mediana di contesto)

Per ogni provincia calcoliamo `quota_critiche = n_sezioni_critiche / n_sezioni`
(la frazione di sezioni eleggibili che supera la soglia del gomito) e la
confrontiamo con `quota_nazionale` calcolata sopra. La mediana del `gap_score`
su tutte le sezioni resta in tabella come indicatore di contesto, ma non e'
piu' il criterio di ordinamento.


In [ ]:
stat_provincia = (
    df.groupby("PROVINCIA")
    .agg(
        n_sezioni=("gap_score", "count"),
        n_sezioni_critiche=("critica", "sum"),
        mediana=("gap_score", "median"),
        min=("gap_score", "min"),
        max=("gap_score", "max"),
    )
)
stat_provincia["quota_critiche"] = stat_provincia["n_sezioni_critiche"] / stat_provincia["n_sezioni"]
stat_provincia = stat_provincia.sort_values("quota_critiche", ascending=False)

print(f"Province con almeno una sezione eleggibile: {len(stat_provincia)}")
print(f"Quota nazionale di sezioni critiche (riferimento): {quota_nazionale:.1%}")
stat_provincia.round(4)


## 4. Grafico riassuntivo: ranking delle province per quota di sezioni critiche

Le province sono ordinate dalla quota piu' alta (la maggior concentrazione
di sezioni critiche) alla piu' bassa. La linea tratteggiata segna la quota
nazionale: le barre a destra di quella soglia sono le province in cui il
fenomeno e' piu' concentrato rispetto alla media italiana.


In [ ]:
ordinata = stat_provincia.sort_values("quota_critiche")
fig, ax = plt.subplots(figsize=(9, max(4, 0.16 * len(ordinata))))

ax.barh(ordinata.index, ordinata["quota_critiche"] * 100, color="#3b6fa0")
ax.axvline(quota_nazionale * 100, color="#c0392b", linestyle="--", linewidth=1,
           label=f"quota nazionale ({quota_nazionale:.1%})")
ax.set_xlabel("% di sezioni critiche sul totale della provincia")
ax.set_title("Quota di sezioni critiche per provincia")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Distribuzione del gap_score, per provincia

Il ranking sopra mostra solo la quota di sezioni critiche: mostriamo quindi,
provincia per provincia, l'istogramma di tutte le sezioni eleggibili, con la
soglia del gomito evidenziata — cosi' si vede quanta parte della coda
destra della distribuzione supera davvero quel livello.

Limitiamo il grafico alle province con almeno `MIN_SEZIONI_PER_GRAFICO`
sezioni eleggibili totali, ordinate per quota di sezioni critiche.


In [ ]:
MIN_SEZIONI_PER_GRAFICO = 5

province_con_grafico = (
    stat_provincia[stat_provincia["n_sezioni"] >= MIN_SEZIONI_PER_GRAFICO]
    .sort_values("quota_critiche", ascending=False)
    .index.tolist()
)
print(f"Province con >= {MIN_SEZIONI_PER_GRAFICO} sezioni eleggibili (istogramma mostrato): "
      f"{len(province_con_grafico)}")

n_col = 6
n_row = int(np.ceil(len(province_con_grafico) / n_col))
fig, axes = plt.subplots(n_row, n_col, figsize=(3 * n_col, 2.2 * n_row), squeeze=False)

for i, provincia in enumerate(province_con_grafico):
    ax = axes[i // n_col][i % n_col]
    valori = df.loc[df["PROVINCIA"] == provincia, "gap_score"]
    ax.hist(valori, bins=30, color="#3b6fa0", edgecolor="none")
    quota = stat_provincia.loc[provincia, "quota_critiche"]
    ax.axvline(SOGLIA_GOMITO, color="#2e7d32", linestyle=":", linewidth=1)
    ax.set_title(f"{provincia}\n(n={len(valori)}, critiche={quota:.1%})", fontsize=8)
    ax.tick_params(labelsize=6)

# spengo le celle vuote della griglia
for j in range(len(province_con_grafico), n_row * n_col):
    axes[j // n_col][j % n_col].axis("off")

fig.suptitle("Distribuzione del gap_score (tutte le sezioni eleggibili), per provincia", y=1.001)
plt.tight_layout()
plt.show()


## 6. Province con pochi dati o senza sezioni critiche

Due controlli complementari alla quota:

- province con **poche sezioni eleggibili in totale** (la quota e' meno
  affidabile: una sola sezione critica su 3 totali da' gia' il 33%);
- province **senza nessuna sezione sopra la soglia del gomito** (quota 0%).


In [ ]:
poche_sezioni = (
    stat_provincia[stat_provincia["n_sezioni"] < MIN_SEZIONI_PER_GRAFICO]
    .sort_values("quota_critiche", ascending=False)
)
print(f"Province con meno di {MIN_SEZIONI_PER_GRAFICO} sezioni eleggibili totali: {len(poche_sezioni)}")
print(poche_sezioni.round(4).to_string() if len(poche_sezioni) else "(nessuna)")

senza_critiche = (
    stat_provincia[stat_provincia["quota_critiche"] == 0]
    .sort_values("n_sezioni", ascending=False)
)
print(f"\nProvince senza nessuna sezione sopra la soglia del gomito: {len(senza_critiche)}")
senza_critiche.round(4)


## 7. Esportazione

Salviamo la tabella `stat_provincia` (quota di sezioni critiche, conteggio
totale, conteggio critiche, mediana, minimo e massimo del gap_score, per
provincia) come base per la reportistica del progetto.


In [ ]:
OUT = CARTELLA_PROGETTO / "quota_sezioni_critiche_per_provincia.csv"
stat_provincia.round(4).to_csv(OUT)
print(f"Salvato: {OUT}")
